# TELEMACHUS — language-agent domain with a larger model (Colab)

Runs the same pipeline as the local GTX 1650 run (Qwen2.5-0.5B) with a larger model:
outcome tables for GSM8K (400 train questions for the learned gate, all 1319 test questions),
then Learn-then-Test certification of every gate (`analysis/llm_eval.py`).

**Runtime -> Change runtime type -> GPU** (T4 is enough for 3B; L4/A100 for 7B).

In [ ]:
# 1) Upload the TELEMACHUS project ZIP
from google.colab import files
import zipfile, os, shutil
from pathlib import Path
up = files.upload()
z = [n for n in up if n.endswith('.zip')][0]
shutil.rmtree('/content/tele', ignore_errors=True)
zipfile.ZipFile(z).extractall('/content/tele')
ROOT = next(p.parent for p in Path('/content/tele').rglob('pyproject.toml') if (p.parent/'src'/'telemachus').exists())
os.chdir(ROOT); print(ROOT)

In [ ]:
# 2) Dependencies
!pip -q install 'transformers>=4.45' accelerate scikit-learn scipy
!nvidia-smi

In [ ]:
# 3) Choose the model. 3B fits a T4; use 7B on L4/A100.
MODEL = 'Qwen/Qwen2.5-3B-Instruct'   # or 'Qwen/Qwen2.5-7B-Instruct'
TAG = MODEL.split('/')[-1]
VOTE_K = 3                            # self-consistency stage (affordable on Colab)
BATCH = 32

In [ ]:
# 4) Outcome tables (resumable: rerun this cell after a disconnect)
import subprocess, sys
env = dict(os.environ, PYTHONPATH='src')
for split, extra in (('train', ['--limit', '400']), ('test', [])):
    subprocess.run([sys.executable, '-m', 'telemachus.llm_agent.generate', '--split', split, *extra,
                    '--batch', str(BATCH), '--vote-k', str(VOTE_K), '--model', MODEL,
                    '--output', f'results/llm_{TAG}/{split}_table.jsonl'], check=True, env=env)

In [ ]:
# 5) Certified evaluation of every gate
import json
sys.path.insert(0, 'src')
from telemachus.llm_agent import cascade
train = cascade.load_table(f'results/llm_{TAG}/train_table.jsonl')
test = cascade.load_table(f'results/llm_{TAG}/test_table.jsonl')
out = {}
for alpha in (.30, .20, .10):
    r = cascade.evaluate(train, test, alpha=alpha, delta=.1, n_cal=600)
    out[f'alpha_{alpha}'] = r
    print('alpha', alpha, {g: (e.get('test') or {}).get('cost') for g, e in r['gates'].items()})
Path(f'results/llm_{TAG}/certified_summary.json').write_text(json.dumps(out, indent=2))

In [ ]:
# 6) Download the results
shutil.make_archive(f'/content/llm_{TAG}', 'zip', f'results/llm_{TAG}')
files.download(f'/content/llm_{TAG}.zip')